<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>RAG with Agent Exercise</center></h1>

# Installing Dependencies

### requirements.txt

openai==0.28  
ipykernel==6.20.2  
langchain==0.0.352  
wikipedia==1.4.0  
rank_bm25==0.2.2   
tiktoken==0.5.2
faiss-cpu==1.7.4

In [3]:
# !pip install -r requirements.txt
# !pip install -r openai==0.28 ipykernel==6.20.2 langchain==0.0.352 wikipedia==1.4.0 rank_bm25==0.2.2 tiktoken==0.5.2 faiss-cpu==1.7.4

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'openai==0.28'


In [4]:
!pip install openai==0.28 ipykernel==6.20.2 langchain==0.0.352 wikipedia==1.4.0 tiktoken==0.5.2
!pip install langchain-openai langchain-community langchain-classic langchain-text-splitters langgraph pypdf
!pip install wikipedia
!pip install faiss-cpu
!pip install rank_bm25

  Using cached openai-0.28.0-py3-none-any.whl.metadata (13 kB)
  Using cached ipykernel-6.20.2-py3-none-any.whl.metadata (6.7 kB)
  Using cached langchain-0.0.352-py3-none-any.whl.metadata (13 kB)
  Using cached tiktoken-0.5.2.tar.gz (32 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
  Using cached langsmith-0.0.92-py3-none-any.whl.metadata (9.9 kB)
  Using cached numpy-1.26.4-cp313-cp313-linux_x86_64.whl
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cach

In [5]:
import getpass
import os

openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

Enter your OpenAI API Key: ··········


`gpt-3.5-turbo` is used as our LLM in this lab

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Collecting movie introduction data from Wikipedia

In [7]:
# https://python.langchain.com/docs/integrations/document_loaders/wikipedia
# Module Wikipedia is required to use WikipediaLoader
from langchain_community.document_loaders import WikipediaLoader

help(WikipediaLoader)

/tmp/ipykernel_1104/2662851850.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


Help on class WikipediaLoader in module langchain_community.document_loaders.wikipedia:

class WikipediaLoader(langchain_core.document_loaders.base.BaseLoader)
 |  WikipediaLoader(
 |      query: str,
 |      lang: str = 'en',
 |      load_max_docs: Optional[int] = 25,
 |      load_all_available_meta: Optional[bool] = False,
 |      doc_content_chars_max: Optional[int] = 4000
 |  )
 |
 |  Load from `Wikipedia`.
 |
 |  The hard limit on the length of the query is 300 for now.
 |
 |  Each wiki page represents one Document.
 |
 |  Method resolution order:
 |      WikipediaLoader
 |      langchain_core.document_loaders.base.BaseLoader
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      query: str,
 |      lang: str = 'en',
 |      load_max_docs: Optional[int] = 25,
 |      load_all_available_meta: Optional[bool] = False,
 |      doc_content_chars_max: Optional[int] = 4000
 |  )
 |      Initializes a new instance of the WikipediaLoader

In [13]:
# load 60 documents from wikipedia
import wikipedia

wikipedia.set_user_agent(
    "RAGBootcamp/1.0 (https://github.com/langchain-ai/langchain)"
)
# load 200 documents from wikipedia
endgame_wikipedia_docs = WikipediaLoader(
    query="Restaurant rating",  # Topic query string
    lang="en",                  # Language code (English)
    load_max_docs=60,           # Set limit to 60 documents as needed
    doc_content_chars_max=10000000,
).load()

In [14]:
len(endgame_wikipedia_docs)

60

In [15]:
endgame_wikipedia_docs[0]

Document(metadata={'title': 'Restaurant rating', 'summary': 'Restaurant ratings identify restaurants according to their quality, using notations such as stars or other symbols, or numbers. Stars are a familiar and popular symbol, with scales of one to three or five stars commonly used. Ratings appear in guide books as well as in the media, typically in newspapers, lifestyle magazines and webzines. Websites featuring consumer-written reviews and ratings are increasingly popular, but are far less reliable.\nIn addition, there are ratings given by public health agencies rating the level of sanitation practiced by an establishment.', 'source': 'https://en.wikipedia.org/wiki/Restaurant_rating'}, page_content='Restaurant ratings identify restaurants according to their quality, using notations such as stars or other symbols, or numbers. Stars are a familiar and popular symbol, with scales of one to three or five stars commonly used. Ratings appear in guide books as well as in the media, typic

# Loading Movie Reviews from a CSV file

In [16]:
# https://python.langchain.com/docs/integrations/document_loaders/csv
from langchain_community.document_loaders import CSVLoader

help(CSVLoader)

Help on class CSVLoader in module langchain_community.document_loaders.csv_loader:

class CSVLoader(langchain_core.document_loaders.base.BaseLoader)
 |  CSVLoader(
 |      file_path: Union[str, pathlib._local.Path],
 |      source_column: Optional[str] = None,
 |      metadata_columns: Sequence[str] = (),
 |      csv_args: Optional[Dict] = None,
 |      encoding: Optional[str] = None,
 |      autodetect_encoding: bool = False,
 |      *,
 |      content_columns: Sequence[str] = ()
 |  )
 |
 |  Load a `CSV` file into a list of `Document` objects.
 |
 |  Each document represents one row of the CSV file. Every row is converted
 |  into a key/value pair and outputted to a new line in the document's
 |  page_content.
 |
 |  The source for each document loaded from csv is set to the value of the
 |  `file_path` argument for all documents by default.
 |  You can override this by setting the `source_column` argument to the
 |  name of a column in the CSV file.
 |  The source of each document w

In [27]:

import pandas as pd
from langchain_community.document_loaders import CSVLoader

file_path = "/content/Yelp Restaurant Reviews.csv"

# 1. قراءة الملف مع الكشف التلقائي عن الفاصل (sep=None)
df = pd.read_csv(file_path, sep=None, engine='python')

# 2. تنظيف أسماء الأعمدة من أي مسافات زائدة
df.columns = df.columns.str.strip()

# 3. طباعة أسماء الأعمدة للتأكد
print("الأعمدة المتاحة حالياً:", df.columns.tolist())

# 4. تحديد آخر عمود (وهو عادة العمود الذي يحتوي على نص التقييم)
text_column = df.columns[-1]

# 5. تحميل المستندات عبر CSVLoader
endgame_csv_docs = CSVLoader(
    file_path=file_path,
    source_column=text_column
).load()

print(f"تم تحميل {len(endgame_csv_docs)} مستند بنجاح!")


الأعمدة المتاحة حالياً: ['Yelp URL,Rating,Date,Review Text']
تم تحميل 19896 مستند بنجاح!


# Spliting documents into chunks
This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

help(RecursiveCharacterTextSplitter)

Help on class RecursiveCharacterTextSplitter in module langchain_text_splitters.character:

class RecursiveCharacterTextSplitter(langchain_text_splitters.base.TextSplitter)
 |  RecursiveCharacterTextSplitter(
 |      separators: 'list[str] | None' = None,
 |      keep_separator: "bool | Literal['start', 'end']" = True,
 |      is_separator_regex: 'bool' = False,
 |      **kwargs: 'Any'
 |  ) -> 'None'
 |
 |  Splitting text by recursively look at characters.
 |
 |  Recursively tries to split by different characters to find one
 |  that works.
 |
 |  Method resolution order:
 |      RecursiveCharacterTextSplitter
 |      langchain_text_splitters.base.TextSplitter
 |      langchain_core.documents.transformers.BaseDocumentTransformer
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      separators: 'list[str] | None' = None,
 |      keep_separator: "bool | Literal['start', 'end']" = True,
 |      is_separator_regex: 'bool' = False,
 |  

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,           # حجم كل مقطع بالنصوص (مثلاً 1000 حرف)
    chunk_overlap=200,          # نسبة التداخل بين الملاحظات لمنع فقدان السياق
    length_function=len,        # دالة حساب الطول (حساب عدد الحروف)
    is_separator_regex=False,   # تحديد ما إذا كانت الفواصل تعابير نمطية Regex أم لا
    separators=["\n\n", "\n", " ", ""]  # الفواصل المستخدمة بالترتيب لتقسيم النص
)

In [30]:
chunked_endgame_wikipedia_docs = text_splitter.split_documents(endgame_wikipedia_docs)
chunked_endgame_csv_docs = text_splitter.split_documents(endgame_csv_docs)
print(
    f"Number of documents in chunked_endgame_wikipedia_docs: {len(chunked_endgame_wikipedia_docs)}"
)
print(
    f"Number of documents in chunked_endgame_csv_docs: {len(chunked_endgame_csv_docs)}"
)

Number of documents in chunked_endgame_wikipedia_docs: 354
Number of documents in chunked_endgame_csv_docs: 23041


# Embedding and Setting Up Storage With FAISS

In [32]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings

# 1. تعريف نموذج الـ Embeddings
embeddings_model = OpenAIEmbeddings(model="text-embedding-ada-002")

# 2. إنشاء التخزين المحلي في مجلد cache
file_store = LocalFileStore("./cache/")

# 3. إعداد الـ CacheBackedEmbeddings
embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embeddings_model,  # النموذج الأساسي لحساب الـ Embeddings
    document_embedding_cache=file_store,     # مكان التخزين المحلي (LocalFileStore)
    namespace=embeddings_model.model,         # اسم النطاق للتمييز بين النماذج (مثلاً: text-embedding-ada-002)
)

/usr/local/lib/python3.13/dist-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [33]:
# Create the vector store with documents:
from langchain_community.vectorstores import FAISS

endgame_csv_db = FAISS.from_documents(chunked_endgame_csv_docs, embedder)
endgame_wiki_db = FAISS.from_documents(chunked_endgame_wikipedia_docs, embedder)

In [34]:
# Set up FAISS retriever:
endgame_csv_faiss_retriever = endgame_csv_db.as_retriever(search_kwargs={"k": 1})
endgame_wiki_faiss_retriever = endgame_wiki_db.as_retriever(search_kwargs={"k": 1})

# Multiple retrievers
In this lab, we will use two retrievers and two sources
1. FAISS retriver
2. BM25 Retriever, a popular ranking function used in information retrieval systems to estimate the relevance of documents to a given search query
    https://python.langchain.com/docs/integrations/retrievers/bm25

Then `Ensemble Retriever` will be used to rerank the results of multiple retrevers in order to achieve better performance.
Sometimes you may want to retrieve documents from multiple different sources, or using multiple different algorithms. The ensemble retriever allows you to easily do this.
https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble

In [35]:
# BM25 retriever:
from langchain_community.retrievers import BM25Retriever

endgame_csv_bm25_retriever = BM25Retriever.from_documents(chunked_endgame_csv_docs, k=1)
endgame_wiki_bm25_retriever = BM25Retriever.from_documents(
    chunked_endgame_wikipedia_docs, k=1
)

In [39]:
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS

# 1. دمج مستندات ويكيبيديا ومستندات الـ CSV معاً
all_docs = endgame_wikipedia_docs + endgame_csv_docs

# 2. تقسيم النصوص باستخدام text_splitter الذي أنشأته سابقاً
docs_splits = text_splitter.split_documents(all_docs)

# 3. إنشاء bm25_retriever و faiss_retriever باستخدام docs_splits
bm25_retriever = BM25Retriever.from_documents(docs_splits)
bm25_retriever.k = 3

vectorstore = FAISS.from_documents(docs_splits, embedder)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [40]:
from langchain_classic.retrievers import EnsembleRetriever

endgame_ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

In [41]:
docs = endgame_ensemble_retriever.invoke("What do people say about the food quality and service?")
docs

[Document(metadata={'source': 'https://www.yelp.com/biz/am%C3%A9lies-french-bakery-and-caf%C3%A9-charlotte-11,5,8/16/2012,"First I would like to begin by saying ""TED is the man!!!"" and \'TEd for PRESIDENT!"". I guess you might be wondering exactly who ""Ted"" is. Well Ted works at Amelie\'s. He provides that type of customer service that is rare these days. Great listener and extreme professional!!! Last week was sort of a rough week for me. (Let\'s just say someone very close to be had a birthday on Tuesday and the anniversary of their death was Wednesday. Always two pretty rough days each year.) Anyway, do their best to try to cheer me up during those rough days. They know I LOVE the Vanilla Chiffon cake at Amelie\'s. When I got home from work I had roses, a cooked dinner and guess what?!! Yep. You guessed it! Vanilla Chiffon Cake from Amelie\'s!!! When it was time to enjoy my cake, I dug right in. For the first time in history, the cake was not good. I wanted to CRY! I thought abo

# Agents
The core idea of agents is to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order.

reference: https://python.langchain.com/docs/modules/agents/

In [42]:
# Tavily(https://tavily.com/) can be used as a search engine to get information online. You have to register to get an API key.
tavily_api_key = getpass.getpass("Enter your Tavily API Key: ")
os.environ["TAVILY_API_KEY"] = tavily_api_key

Enter your Tavily API Key: ··········


In [43]:
# https://docs.tavily.com/docs/tavily-api/langchain
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

# 2. إعداد المحرك والأداة
search = TavilySearchAPIWrapper()
tavily_tool = TavilySearchResults(api_wrapper=search)

# 3. تشغيل الأداة
# استعلام عن أفضل تقييمات المطاعم
tavily_tool.run("top rated restaurant reviews and ratings")

/tmp/ipykernel_1104/3891320876.py:7: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(api_wrapper=search)


[{'title': 'How Restaurant Review Websites Affect Consumer Demand and Spend',
  'url': 'https://pos.toasttab.com/blog/on-the-line/restaurant-reviews-and-ratings-data',
  'content': '### 46% of diners are likely to check Google Reviews first\n\nSo where are these ratings coming from? For 46% of people, Google Reviews is the first place they check restaurant ratings, while 23% check ratings on Yelp first. This comes as no surprise as users can immediately see your Google Reviews when they search for your restaurant.\n\nBeyond these top two sources, users are also checking restaurant ratings on a few other sites. 9% first check TripAdvisor, 6% look at OpenTable, and 3% use Resy. However, 10% answered that they don’t use any of these sites. [...] More specifically, females are more sensitive to lower ratings and have a higher threshold. For example, if a restaurant has less than a 3.5-star rating, women aren’t as likely to try it. Whereas males are more likely to try a lower-rated restaura

In [44]:
from langchain_core.tools.retriever import create_retriever_tool

endgame_docs_retrieval_tool = create_retriever_tool(
    retriever=endgame_ensemble_retriever,
    name="restaurant_reviews_retriever",
    description="Searches and retrieves information about restaurant reviews, ratings, food quality, and customer service."
)

endgame_retriever_tools = [endgame_docs_retrieval_tool, tavily_tool]

Tools have been created so we are ready to build an agent with them.

In [45]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# Replace create_conversational_retrieval_agent with LangGraph's create_react_agent
endgame_agent_executor = create_react_agent(llm, endgame_retriever_tools)


/tmp/ipykernel_1104/1726291271.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  endgame_agent_executor = create_react_agent(llm, endgame_retriever_tools)


In [47]:
# Use .invoke() instead of calling directly
response = endgame_agent_executor.invoke(
    {"messages": [HumanMessage(content="What are the most common customer complaints regarding food and service?")]}
)

print(response["messages"][-1].content)

The most common customer complaints regarding food and service include issues with customer service, pricing, quality of food, wait times, and employee behavior. Customers have expressed dissatisfaction with service that is perceived as slow, unresponsive, or rude. Additionally, complaints about the quality of food, pricing being too high for the value received, and long wait times have been highlighted in reviews. In some cases, customers have also mentioned disagreements among employees and inefficiencies in the service process.


In [48]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. إنشاء كائن للحفظ في الذاكرة
message_history = ChatMessageHistory()

# 2. مغلف Agent مع إدارة السجل
endgame_agent_with_chat_history = RunnableWithMessageHistory(
    runnable=endgame_agent_executor,
    get_session_history=lambda session_id: message_history,
    input_messages_key="messages",
    history_messages_key="chat_history",
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [49]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"session_id": "1"}}

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content="What are the most common customer complaints regarding food and service?")]},
    config=config
)

print(response["messages"][-1].content)

The most common customer complaints regarding food and service include issues with customer service, pricing, quality of food, wait times, and employee behavior. Customers have expressed dissatisfaction with service quality, pricing being too high for the average person, long wait times, and mediocre quality of products. Additionally, complaints about employees bickering about tips and inefficient processes have been mentioned.


In [50]:
from langchain_core.messages import HumanMessage

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content="How did the management respond to these complaints?")]},
    config={"configurable": {"session_id": "2"}}
)

print(response["messages"][-1].content)

Here are some responses from the management to the complaints:

1. Carlos Bakery Las Vegas:
   - The customer complained about the poor response time to email orders and the shortage of staff. The management was criticized for being in training mode and incapable of dealing with large volume orders and crowds. The customer advised waiting a few more weeks before visiting the bakery. The management was described as poor, and the customer suggested not emailing cake requests but mentioned that pre-made cakes turned out well.

2. Amélie's French Bakery and Café Charlotte:
   - The customer complained about the pretentious management and the lack of seating. The management's response to the customer's request for a place to sit was perceived as disrespectful. The customer also mentioned the management's practices with back-of-the-house employees, such as requiring them to work long hours without proper compensation. The customer expressed disappointment in the management's lack of respect 

In [51]:
from langchain_core.messages import HumanMessage

response = endgame_agent_with_chat_history.invoke(
    {"messages": [HumanMessage(content="What do customer reviews say about the food quality and atmosphere?")]},
    config={"configurable": {"session_id": "3"}}
)

print(response["messages"][-1].content)

Here are some customer reviews about food quality and atmosphere from different restaurants:

1. Carlos Bakery in Las Vegas:
   - Rating: 4 stars
   - Review: Customers love the fresh food made daily and the good customer service. The atmosphere is also praised, with a hint of a better taste compared to other places. Customers have been visiting this place for over 6 years and always enjoy their experience.

2. Amélie's French Bakery and Café in Charlotte:
   - Rating: 4 stars
   - Review: The atmosphere and desserts are great, but there might be a long line and wait. The food is worth the wait, with many desserts to choose from. The limited seating and space create an artsy atmosphere in the Noda area.

3. Page's in Pittsburgh:
   - Rating: 5 stars
   - Review: Customers enjoy the friendly atmosphere and delicious ice cream. The outstanding customer service and comfortable atmosphere make it a pleasant experience. Outdoor seating is available, and the prices are affordable.

These rev